# **Introduction (Assignment 1, Deadline 23/10/2026 at 08:00 AM)**

Students write text here, explaining:
 - What it means to preprocess data and what is a preprocessing pipeline
 - What are the main artefacts that can be found in the TMS-EEG signal and their causes
 - **Optional**: a personal comment or opinion on the problems posed by preprocessing, based on what you heard in class

**Answer:**

In TMS-EEG, preprocessing describes the process of seperating the signal, which is the fraction of scalp voltage generated by the cerebral cortex in response to the TMS pulse, from the noise, namely the fraction of scalp voltage generated by anything else. 
A preprocessing pipeline is the sequence of transformations applied to the data.
One of the most common artefacts that can be found in TMS-EEG, are muscle artefacts. If TMS is done over the scalp area with large muscles, they most probably gave huge muscle artefacts because TMS activates everything sensitive to magnetism within reach. Another common aretfact could occur when stimulating more frontally, which leads the subject to blink moe frequently than they would under normal conditions. Of course the TMS-pulse itself generates a huge artefact when picked up by the EEG-electrodes over the scalp, so researchers also have to account for this in their preprocessing. 


The problem posed by preprocessing is, that there is no stamdard preprocessing pipeline for EEG or TMS-EEG. What follows is, that every research group all around the world has a different preprocessing pipeline. 
So far, there is also no common ground for the basis on which preprocessing is done. Some groups aim to apply as little transformations to the data as possible in order to avoid changing the data in undesirable or destructive ways. They rather take the risk of having too much noise in their data, tham to loose anything. 
Another opposing approach is to act under the assumption that the actual signal is invisible until all artefacts have been removed. Researchers that believe in this philosophy rather take the risk of deleting some brain activity in order to delete non-brain information, so artefacts, enitrely. 

Generally, two research groups could analyze the same data but get to different results due to completely opposing preprocessing philosophies. This example highlights the importance of documenting all steps in the preprocessing pipeline, when publishing results. 



In [1]:
from pathlib import Path                            # to build a bridge between Python and the filesystem 

import numpy as np                                  # to perform array operations
import matplotlib                                   # this and the following to draw plots
import matplotlib.pyplot as plt
matplotlib.use('Qt5Agg')                            # to make plots interactive


import mne                                          # to read and manipulate EEG data

from scripts import utils                           # custom functions to shorten the code in this notebook

Channels marked as bad:
none


# **1. Basic Preprocessing (Assignment 1, Deadline 23/10/2026 at 08:00 AM)**

Students write text here, listing the operations that we categorise as "basic preprocessing".

If you cannot remember what the steps are, look at the code in the cells below and you will recall. 

**Answer:**

Under basic preprocessing we understand five principle operations. 
**Loading the data**, meaning making it available for the software we want to analyze the data with.

**Inspecting the data** to see its shape and content. Shape describes the number of channels and time-length, resulting in a 2D data format. We also have acquisition parameters, informing about the sampling rate and improtant to understand the content of the data. When inspecting the data, one can also comment on its appearance, informing how visibly dirty the raw data is and to what extent. 

In EEG studies, researchers often coregister data that is not directly related to the actual EEG they are measuring, for example Electromyography, Electrooculogram or Electrocardiogram. When **adjusting the data**, one deletes non-EEG channels that were coregistered.

The other two operations are the **Interpolation of the pulse artefact** and **High-pass Filtering**

## **1.1. Loading data**

**Optional but welcome:** students write here any considerations about this step.

**Answer:**

The data evaluated in this notebook is from a single, healthy subject: **S02C1_M1**.
 The relevant information is distributed across three different files. In this step, first  a path object is defined, which makes the data directory visible for python; then the subject's **vhdr** file is explicitly loaded and saved under the variable "file_of_interest". This file contains metadata, which is all the improtant information on how the data was acquired. 

In [2]:
#defines a path object, make data directory visible for python 
data_dir = Path("data")
subject_and_condition = "S02C1_M1" #name of subject and experimental conditions assigned to this variable, can use it then in following line 

file_of_interest = list(data_dir.glob(pattern=f"{subject_and_condition}.vhdr")) # we are reading the .vhdr file not the actual eeg data
print(file_of_interest, "|", type(file_of_interest), "|", file_of_interest[0])

[PosixPath('data/S02C1_M1.vhdr')] | <class 'list'> | data/S02C1_M1.vhdr


In [3]:
eeg_data = mne.io.read_raw_brainvision(vhdr_fname=file_of_interest[0])

Extracting parameters from data/S02C1_M1.vhdr...
Setting channel info structure...


## **1.2. Adjusting the data**

**Compulsory:** students write here the functional significance of this step (that is, why we do it).

**Answer:**

During the EEG recordings, data from auxiliary channels is often recorded simultaneously, for example electrodes around the eyes to record the electrical activity given my eye blinks of the subject, or electrodes on the wrist to record electrocardiagraphy. 
If one is interested in an EEG-only analysis, any non-EEg channels need to be removed. 
If these channels would not be removed, they might influence the anaylsis and ultimatly lead to false results. 

In [4]:
channels_to_drop = []

for channel_name in eeg_data.ch_names: #contains names of all channels, for every channel we do the operation
    try:
        int(channel_name[-1]) #check wether last letter an integer number or not, if it is, nothing done 
    except ValueError: #if it fails
        if channel_name[-1] != "z": #if it is a letter it can either be the z or the g or i
            channels_to_drop.append(channel_name) # so if its not a z, you take it and append it to a list called channels to drop
print(f"Channels to drop: {channels_to_drop}") #list of channels that we want to drop 

eeg_data = eeg_data.drop_channels(ch_names=channels_to_drop) #drops the ones we dont want and makes a new list with all the eeg channels 
#can print it again then and check wether they were actually removed 

Channels to drop: ['EOG', 'FDI']


Here, electrooculogram (EOG) and electromyography from the first dorsal interosseous (FDI) have been co-recorded. With the code block above, these channels have been dropped from the data set, leaving only actual the EEG-channels to analyze.

## **1.3.Inspecting and Visualizing the data**

**Compulsory:** students write here the functional significance of this step (that is, why we do it).

**Answer:**

In this step, the data is inspected and channel locations are set. The table provides basic information about the acqusition, channels, filters and general variables like the filename, date and object type.

As can be told by the table below,the data contains no information about the shape of the head nor the position of the electrodes on it. There is huge inter-subject variability in head shape and size, a channel can end up on very different places, so we need to put the channels in relation to the head. 
Since the amount of channels is known, MNE's built-in 'easycap-M1' montage can be used and the 3D coordinates of each channel can be projected on a 2-dimensional plane, visualizing their positions.
Projecting the channels onto an average standard headhsape is fundamental for source reconstruction and visulaization. 

In [8]:
easycap_m1_montage = mne.channels.make_standard_montage(kind="easycap-M1") 
easycap_m1_montage.plot()
eeg_data.set_montage(montage=easycap_m1_montage,
                     on_missing="ignore")

<RawBrainVision | S02C1_M1.eeg, 70 x 3307600 (661.5 s), ~86 KiB, data not loaded>

**Compulsory:** here, students describe and comment the plot generated by the code below. In particular, I would like to read:

- What each row represents
- What are the $x$ and $y$ of each row, and their units of measurement
- A description of the artefacts you see, and their likely causes

**Answer:**

The plot generated by the line of code below shows the 70 EEG-channels on its $y$-axis, named after the typical 10-20 system, and time in seconds on the $x$-axis. Therefore, each row represents the elcetrical activity picked up by the corresponding electrode over time. 


3 artefacts are very noticable in the plot:
The first one is repeats itself around every second and is most prominent in the frontal channels. It is much larger in amplitude than the brain activity and lasts only shortly, around 100 ms. These characteristics indicate that we might deal with **eyeblinks**. 
The second one is very prominent across all channels. It has a very high amplitude and frequency, which makes it show up as thick black lines in our plot. The fact that it affects many channels, indicates that it spreads across space. Most likely, this artefact is a **muscle artefact**, since muscles are relatively large. 
Futhermore, there is a a noticable **TMS-pulse artefact**, which appears exactly at the same time as the temporal marker, that marks stumulus onset. 

Additionally, channel 'FC3' continuously shows very high frequency, which might also be an artefact.

In [ ]:
eeg_data.plot()

**Compulsory:** here, students define event markers and explain why they are important for EEG data analysis

**Optional:** any comments that go beyond a mere definition are welcome. For example, you could make comments about the structure of event arrays and what it tells us about the nature of events

**Answer:**

Event markers are essential for identifying and segmenting key moments during EEG data collection, for example the TMS-pulse. They are time-stamps that locate events of experimental interest and are usually defined as a set of parameters. 

Event arrays capture these parameters. For TMS-EEG, there is an ordinal number which infroms on the number of pulse that was given. They have a type and a name, which can be either a word or a numeric code, chosen by the researchers. 
There is also the type of occurance stored in units of data points, which means their positions in the EEG matrix; and for how many data points the stimulus lasts. 
Lastly, the number of channels that are affected is stored aswell.


In [7]:
events_from_annotations, events_dict = mne.events_from_annotations(raw=eeg_data) #take that function for use, takes raw data as inpiut and returns 2 things 
events_from_annotations = events_from_annotations[2:]
#can identify TMS pulses on the trace and interpolate 

Used Annotations descriptions: ['New Segment/', 'Stimulus/S 54', 'Stimulus/S255']


## **1.4. Interpolating the pulse artefact**

**Compulsory:** students write here the functional significance of this step (that is, why we do it).

**Answer:**

Once the artifact is identified, it needs to be get rid of. Under the TMS pulse, there would be information generated by the brain, but it is inaccessible due to the TMS pulse hiding it. In this step, the TMS pulse artefact is cut out and the resulting gap is closed via interpolation. 
Interpolation is the act of estimating a function in a given interval, given values from preceeding interval and a following one. Essentially, there is missing data under the TMS pulse artefact and it can be reconstructed via interpolation. 

In [8]:
#divides eeg trace into blocks, centered on the TMS pulses and making a plot 
pre_interpolation_epochs = mne.Epochs(raw=eeg_data,
                                      events=events_from_annotations,
                                      tmin=-1.1,
                                      tmax=0.5,
                                      baseline=None)
pre_interpolation_epochs_tep = pre_interpolation_epochs.average()
pre_interpolation_epochs_tep.plot();

Not setting metadata
200 matching events found
No baseline correction applied
0 projection items activated


In [ ]:
# file will contain huge block of EEG data but there are noit the same TMS pulse artefacts, since its already interpolated 
post_interpolation_eeg = mne.io.read_raw(fname="data/post_2_5_interpolation_eeg.fif", 
                                         preload=True)
post_interpolation_epochs = mne.Epochs(raw=post_interpolation_eeg,
                                       events=events_from_annotations,
                                       tmin=-1.1,
                                       tmax=0.5,
                                       baseline=None)
post_interpolation_tep = post_interpolation_epochs.average()
post_interpolation_tep.plot(); #plot to show the results 
#can still see huge fluctations but we got rid of the main artifact 

Opening raw data file data/post_2_5_interpolation_eeg.fif...
    Range : 0 ... 3307599 =      0.000 ...   661.520 secs
Ready.
Reading 0 ... 3307599  =      0.000 ...   661.520 secs...
Not setting metadata
200 matching events found
No baseline correction applied
0 projection items activated


## **1.5. High-pass Filtering**

**Compulsory:** students write here the functional significance of this step 

**Answer:**

EEG signals are a combination of many oscillations at many different frequencies. Whereas some oscillations are genuine brain activity, others represent artefactual sources. For example, sweating of the subject causes very slow oscillations, which should be filtered out. 
A high-pass filter filters the data by leaving out everything that oscillates at a lower frequency than a given threshold, which in this case is 0.1 Hz. Everything that oscillates at a higher frequency than those 0.1 Hz is allowed to pass through. 

In [10]:
#TEP looks like after interpolating the artefact and filtering to keep anything above 0.1 Hz 
#interpolizes high pass filter, that filters data that leaves out everything oscilaating lower than 01. Hz 
post_filtering_eeg = mne.filter.filter_data(data=post_interpolation_eeg.get_data(),
                                            sfreq=post_interpolation_eeg.info["sfreq"],
                                            l_freq=0.1, # our cut-off frequency 
                                            h_freq=None,
                                            method="iir",
                                            iir_params=None,
                                            copy=True,
                                            phase="zero")
post_filtering_eeg = mne.io.RawArray(data=post_filtering_eeg,
                                     info=post_interpolation_eeg.info)
post_filtering_epochs = mne.Epochs(raw=post_filtering_eeg,
                                   events=events_from_annotations,
                                   tmin=-1.1,
                                   tmax=0.5,
                                   baseline=None)
post_filtering_tep = post_filtering_epochs.average()
post_filtering_tep.plot();

Setting up high-pass filter at 0.1 Hz

IIR filter parameters
---------------------
Butterworth highpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 8 (effective, after forward-backward)
- Cutoff at 0.10 Hz: -6.02 dB

Creating RawArray with float64 data, n_channels=70, n_times=3307600
    Range : 0 ... 3307599 =      0.000 ...   661.520 secs
Ready.
Not setting metadata
200 matching events found
No baseline correction applied
0 projection items activated
